##### Copyright 2025 Google LLC.

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

### Data Sources & Licensing
This notebook uses modified sample documents for educational purposes.

*   **Files:** `Jesse Nathan_1.pdf`, `Jesse Nathan_2.pdf`, `Jesse Nathan_3.pdf`
*   **Original Source:** [Scribd_templates](https://www.scribd.com/),
*   **Modification Note:** These files were modified by the author to include specific financial profiles for this exercise.
*   **License:** The original files are used under CC-BY. Modifications are licensed under [Apache 2.0](https://apache.org).


# Customer financial profiler

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/examples/customer_financial_profiler/Customer_financial_profiler.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

<!-- Community Contributor Badge -->
<table>
  <tr>
    <!-- Author Avatar Cell -->
    <td bgcolor="#d7e6ff">
      <a href="https://github.com/sharathrushi" target="_blank" title="View Sharath Rushi's profile on GitHub">
        <img src="https://github.com/sharathrushi.png?size=100"
             alt="Sharath's GitHub avatar"
             width="100"
             height="100">
      </a>
    </td>
    <!-- Text Content Cell -->
    <td bgcolor="#d7e6ff">
      <h2><font color='black'>This notebook was contributed by <a href="https://github.com/sharathrushi" target="_blank"><font color='#217bfe'><strong>Sharath Rushi</strong></font></a>.</font></h2>
      <h5><font color='black'><a href="https://github.com/sharathrushi" target="_blank">
      <!-- Footer -->
      <font color='black'><small><em>Have a cool Gemini example? Feel free to <a href="https://github.com/google-gemini/cookbook/blob/main/CONTRIBUTING.md" target="_blank"><font color="#078efb">share it too</font></a>!</em></small></font>
    </td>
  </tr>
</table>

# Overview
Estimate a customer's financial status using their financial documents (e.g., payslips, rental agreements, house valuations, and share statements)

In this tutorial, you will:
* Estimate monthly income
* Identify movable and immovable assets

This process helps financial institutions assess loan eligibility and estimate a customer's total value.

## Setup

### Install SDK

In [ ]:
%pip install -U -q 'google-genai>=1.0.0'  # Install the Python SDK

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.7/793.7 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.5/245.5 kB 13.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.51.0 which is incompatible.


In [ ]:
# This step might ask you to restart the session for the installed packages to be reflected
# Versions other than specified are throwing errors for albumentations
%pip install -U -q pymupdf nougat-ocr transformers "albumentations==1.3.1"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.7/125.7 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.5/82.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.5/431.5 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 83.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 92.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

### Set up your API key

To run the following cell, your API key must be stored it in a Colab Secret named `GOOGLE_API_KEY`. If you don't already have an API key, or you're not sure how to create a Colab Secret, see the [Authentication ![image](https://storage.googleapis.com/generativeai-downloads/images/colab_icon16.png)](../quickstarts/Authentication.ipynb) quickstart for an example.

In [ ]:
from google.colab import userdata
from google import genai
from google.genai import types

GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
client = genai.Client(api_key=GOOGLE_API_KEY)

Now select the model you want to use in this guide, either by selecting one in the list or writing it down. Keep in mind that some models, like the 2.5 ones are thinking models and thus take slightly more time to respond (cf. [thinking notebook](./Get_started_thinking.ipynb) for more details and in particular learn how to switch the thinking off).

In [ ]:
MODEL_ID = "gemini-2.5-flash" # @param ["gemini-2.5-flash-lite", "gemini-2.5-flash", "gemini-2.5-pro", "gemini-3.1-flash-lite-preview", "gemini-3-flash-preview", "gemini-3.1-pro-preview"] {"allow-input":true, isTemplate: true}

# Configurable parameters

In [ ]:
# Change this to the person you are interested in and upload their financial documents
person_of_interest = "Jesse Nathan" # @param {type:"string"}

# Load the necessary sample files

##### Download the sample input files to session storage.
##### Note: These files are deleted after the session expires.

In [ ]:
import os

DATA_DIR = "."
RAW_URL = "https://raw.githubusercontent.com/google-gemini/cookbook/main/examples/customer_financial_profiler/data/"

files = ["Jesse Nathan_1.pdf", "Jesse Nathan_2.pdf", "Jesse Nathan_3.pdf"]

for f in files:
    if not os.path.exists(os.path.join(DATA_DIR, f)):
        !wget -q {RAW_URL + f.replace(' ', '%20')} -P {DATA_DIR}

print(f"Verified files in root: {[f for f in os.listdir('.') if f.endswith('.pdf')]}")

Verified files in root: ['Jesse Nathan_2.pdf', 'Jesse Nathan_3.pdf', 'Jesse Nathan_1.pdf']


# Imports and files

In [ ]:
import torch
from transformers import AutoProcessor, VisionEncoderDecoderModel

device = "cuda" if torch.cuda.is_available() else "cpu"
all_files = os.listdir('.')
person_of_interest_files = [f for f in all_files if f.startswith(person_of_interest)]
nougat_model_id = "facebook/nougat-small"
processor = AutoProcessor.from_pretrained(nougat_model_id, do_crop_margin=False)
model_base = VisionEncoderDecoderModel.from_pretrained(nougat_model_id).to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/479 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/484 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/165 [00:00<?, ?B/s]

# Perform AI/ML checks
Check whether the person is linked to illegal activities, terrorism or money laundering

In [ ]:
# @title Helper function to add citations
def add_citations(response):
    text = response.text

    # Check if grounding metadata exists and has supports/chunks
    if not response.candidates or not response.candidates[0].grounding_metadata:
        print("No grounding metadata available. Returning original text.")
        return text

    grounding_metadata = response.candidates[0].grounding_metadata
    supports = grounding_metadata.grounding_supports
    chunks = grounding_metadata.grounding_chunks

    if not supports or not chunks:
        print("No grounding supports or chunks available. Returning original text.")
        return text

    # Sort supports by end_index in descending order to avoid shifting issues when inserting.
    sorted_supports = sorted(supports, key=lambda s: s.segment.end_index, reverse=True)

    for support in sorted_supports:
        end_index = support.segment.end_index
        if support.grounding_chunk_indices:
            # Create citation string like [1](link1)[2](link2)
            citation_links = []
            for i in support.grounding_chunk_indices:
                if i < len(chunks):
                    uri = chunks[i].web.uri
                    citation_links.append(f"[{i + 1}]({uri})")

            citation_string = ", ".join(citation_links)
            text = text[:end_index] + citation_string + text[end_index:]

    return text

In [ ]:
# @title Helper function for AML screening
from pydantic import BaseModel

# Define the schema for structured output
class ScreeningResult(BaseModel):
    is_involved: bool


# Gemini API does not support using response_mime_type: 'application/json' (Structured Output)
# with Tools (such as Google Search Grounding) in the same request
# Hence, a 2 step approach is used.


def aml_screening_with_grounding(name):
    # --- STEP 1: Search for information (No JSON mode) ---
    grounding_tool = types.Tool(google_search=types.GoogleSearch())
    search_config = types.GenerateContentConfig(tools=[grounding_tool])

    search_query = f"""
        Search for recent news and publicly available information regarding {name}.
        Is there any credible indication they are involved in money laundering,
        terrorism financing, or other illegal activities?
    """
    search_response = client.models.generate_content(
        model=MODEL_ID,
        contents=search_query,
        config=search_config,
    )
    factual_text = search_response.text

    text_with_citations = add_citations(search_response)
    print(f"Grounding analysis for {name}:\n{text_with_citations}")


    # --- STEP 2: Extract Structured Boolean (JSON mode) ---
    # Pass the search results into the prompt for the second call

    extraction_config = types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=ScreeningResult,
    )
    extraction_prompt = f"""
        Based ONLY on the following research results,
        determine if the {name} is involved in illegal activities.
        Research Results:
        {factual_text}
    """

    final_response = client.models.generate_content(
        model=MODEL_ID,
        contents=extraction_prompt,
        config=extraction_config,
    )
    # Use the .parsed attribute to get the boolean safely
    result = final_response.parsed

    print(f"Verified result for {name}: {result.is_involved}")

    return result.is_involved


In [ ]:
red_flags = aml_screening_with_grounding(person_of_interest)
print(f"Red flags for {person_of_interest}: {red_flags}")

Grounding analysis for Jesse Nathan:
Based on publicly available information, there is no credible indication that Jesse Nathan, the poet and lecturer, is involved in money laundering, terrorism financing, or other illegal activities.

Recent news and publicly available information predominantly focus on Jesse Nathan's literary and academic career. He is recognized as a poet whose works have appeared in notable publications such as *Paris Review*, *Kenyon Review*, *The Yale Review*, *The Nation*, and *American Poetry Review*[1](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEpJDsw9qoFnmC3l11NAxUDjY1jJS_b34XVl_lo6WMRTj_vJytLHGCYE7bEAeYrtkYgTTPqArYzcBvr_wMpx1AFu5ylXm1XAGEqEoMoXJwjzGoax__72WTwMDXPF4ODqX9WL_RWDgzm821WuqD60_pbMjHdtiKOeZ2mwa-E6j2cuh1c2PcG-H4KmM66DbeUa3o928WWTAE=). He holds a PhD in literature from Stanford and is a lecturer in the English Department at UC Berkeley[1](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEpJDsw9qoFnmC3l11N

# Process source of wealth files
##### Load the PDF, extract text, and generate the financial profile of the customer.
##### Note: This also supports image or scanned PDF documents.

In [ ]:
# @title Helper function to extract text from PDF
import pymupdf as fitz
from PIL import Image

def pdf_to_text(pdf_path):
    doc = fitz.open(pdf_path)
    full_text = ""
    for page_num, page in enumerate(doc):
        page_text_fitz = page.get_text()

        # If fitz extracts no text, assume it's an image-based PDF and use Nougat
        if not page_text_fitz.strip():
            # Render page to an image (high resolution for OCR)
            # Using a matrix to increase resolution (e.g., 2x2 or 3x3) can improve OCR accuracy.
            pix = page.get_pixmap(matrix=fitz.Matrix(2, 2)) # 2x resolution
            img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)

            # Process image with Nougat
            pixel_values = processor(
                images=img,
                return_tensors="pt",
                do_crop_margin=False,
                do_thumbnail=False,
                do_align_long_axis=False
            ).pixel_values.to(device)

            # Ensure pixel_values are float32 for the model
            if pixel_values.dtype != torch.float32:
                pixel_values = pixel_values.float()

            # Generate text using the Nougat model. Removed decoder_input_ids as it's not needed for starting generation from image.
            outputs = model_base.generate(
                pixel_values,
                min_length=1,
                max_new_tokens=1024, # Reduced max_new_tokens to prevent index out of range error
                bad_words_ids=[[processor.tokenizer.unk_token_id]],
            )
            page_text_ocr = processor.batch_decode(outputs, skip_special_tokens=True)[0]
            full_text += page_text_ocr + "\n\n" # Add some separation between pages
        else:
            full_text += page_text_fitz + "\n\n" # Add some separation between pages
    return full_text

In [ ]:
from typing import List

class MonthlyIncome(BaseModel):
    value: float
    currency: str
    period: str

class ImmovableAsset(BaseModel):
    description: str
    value: float
    currency: str
    type: str
    valuation_basis: str
    age_of_property_years: float

class Stock(BaseModel):
    company_name: str
    number_of_shares: float
    par_value_per_share: float
    currency_per_share: str
    total_value: float
    total_value_currency: str
    owner: str

class Cash(BaseModel):
    value: float
    currency: str
    note: str

class MovableAssets(BaseModel):
    stocks: List[Stock]
    cash: Cash

class AggregatedValues(BaseModel):
    monthly_income: MonthlyIncome
    immovable_assets: List[ImmovableAsset]
    movable_assets: MovableAssets

class FinancialProfile(BaseModel):
    is_primarily_about_target: bool
    aggregated_values: AggregatedValues

In [ ]:
# @title Helper function to verify and extract information
import json
def verify_and_extract(text, target_name):
    prompt = f"""
        Analyze the following text from multiple sources regarding {target_name}.
        1. Verify if all sources document is primarily about {target_name}.
        2. Aggregate the values to retrieve the following information:
            - Monthly Income
            - Immovable Assets (Real Estate)
            - Movable Assets (Cash/Stocks)
        3. Format as JSON.
        Text: {text}
    """
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
        config={
            "response_mime_type": "application/json",
            "response_schema": FinancialProfile,
        },
    )
    return response.parsed

In [ ]:
# While native PDF support is available for Gemini models
# this implementation acts as a guide for users building modular pipelines
# that combine local specialized models with Gemini’s reasoning capabilities
from pydantic.json import pydantic_encoder
combined_text = ""
for file in person_of_interest_files:
    text = pdf_to_text(file)
    print(f"Extracting information from {file}")
    combined_text += text + "\n"
result = verify_and_extract(combined_text, person_of_interest)
print(json.dumps(result, indent=4, default=pydantic_encoder))

Extracting information from Jesse Nathan_2.pdf
Extracting information from Jesse Nathan_3.pdf
Extracting information from Jesse Nathan_1.pdf
{
    "is_primarily_about_target": true,
    "aggregated_values": {
        "monthly_income": {
            "value": 8000,
            "currency": "USD",
            "period": "monthly"
        },
        "immovable_assets": [],
        "movable_assets": {
            "stocks": [],
            "cash": {
                "value": 0,
                "currency": "USD",
                "note": "No explicit cash assets found in this document."
            }
        }
    }
}


## Next steps
1. Token & Cost Management

    Token Counts: Use client.model.count_tokens(prompt) before calling the API.

    Billing: Gemini 1.5 Flash is significantly cheaper than Pro. For high-volume PDF parsing, use Flash.

    Reduction: Implement Semantic Chunking. Instead of sending a 100-page PDF, use a lightweight NLP tool (like spaCy) to find pages containing "Balance Sheet" or "Assets" and only send those pages to the LLM.

2. Metrics & Benchmarking
| Metric | Description |
| :--- | :--- |
| Extraction Accuracy | Compare LLM output against a manually labeled "Golden Dataset." |
| Latency | Time taken from PDF upload to final report (Target: < 30s). |
| F1 Score | For NER (Named Entity Recognition) of asset values. |
3. Hallucination Check & Monitoring

    Self-Reflection: Ask the model to "Provide the exact quote from the text where you found this asset value." If it can't, flag it as a potential hallucination.

    NLI (Natural Language Inference): Use a smaller model to check if the generated "Summary" is logically entailed by the "Source Text."

    Monitoring: Use tools like Arize Phoenix or LangSmith to track drift and "faithfulness" scores in production.

4. Human-in-the-Loop (HITL)

In financial compliance, AI should never make the final "Reject" decision.

    Confidence Thresholds: If the LLM confidence is <0.85, route the case to a compliance officer.

    UI Annotation: Highlight the source PDF text in a dashboard so the human can quickly verify the AI's extraction.

5. NLP Alternatives (Non-GenAI)

To reduce costs or increase speed, use:

    Entity Extraction: spaCy or Hugging Face FinBERT (fine-tuned for finance).

    Pattern Matching: Regular Expressions (Regex) for specific formats like CIBIL scores or Currency amounts.

    Classification: XGBoost on TF-IDF vectors to categorize documents before they reach the LLM.